<a href="https://colab.research.google.com/github/dewzpruth/genaiexps/blob/main/5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from datasets import load_dataset
from transformers import (GPT2LMHeadModel, GPT2Tokenizer, Trainer,
 TrainingArguments, DataCollatorForLanguageModeling)
import matplotlib.pyplot as plt
# Use tiny sample dataset (only 200 examples as per your requirement - no huge dataset)
print("Using device: cuda")
dataset = load_dataset("amazon_polarity", split="train[:200]")
print(f"Dataset loaded with {len(dataset)} examples")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id
def tokenize_function(examples):
 texts = [t + "" + c for t, c in zip(examples['title'], examples['content'])]
 return tokenizer(texts, truncation=True, max_length=256, padding="max_length")
tokenized = dataset.map(tokenize_function, batched=True,
 remove_columns=dataset.column_names).train_test_split(0.1)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("Tokenization completed.")
print("Starting fine-tuning of GPT-2 on product reviews...")
training_args = TrainingArguments(
 output_dir="./gpt2-finetuned", num_train_epochs=3,
 per_device_train_batch_size=8, eval_strategy="epoch",
 learning_rate=5e-5, weight_decay=0.01, report_to="none")
trainer = Trainer(model=model, args=training_args,
 train_dataset=tokenized["train"], eval_dataset=tokenized["test"],
 data_collator=data_collator)
trainer.train()
trainer.save_model("./gpt2-finetuned")
print("Fine-tuning completed and model saved!")
# === SHOW GENERATED PRODUCT REVIEWS (as per lab manual) ===
print("\n=== Generated Product Reviews ===")
prompt = "This wireless earbuds are"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
 outputs = model.generate(**inputs, max_new_tokens=80,
 temperature=0.8, top_p=0.9, do_sample=True,
 repetition_penalty=1.2)
generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Prompt: This wireless earbuds are")
print(f"Generated: {generated}")
print("Fine-tuning completed and model saved!")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

amazon_polarity/train-00000-of-00004.par(…):   0%|          | 0.00/260M [00:00<?, ?B/s]

amazon_polarity/train-00001-of-00004.par(…):   0%|          | 0.00/258M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Fine-tuning completed and model saved!

=== Generated Product Reviews ===
Prompt: This wireless earbuds are
Generated: This wireless earbuds are great for music lovers, and you get a lot of good sound quality. However the only thing that really annoys me is when I use my iPhone on it while playing songs or listening to podcasts - just because they play audio tracks instead of an actual song makes them difficult to hear in these headphones (unless your ears can make out what's going through each channel). Also like all this Bluetooth speaker stuff
Fine-tuning completed and model saved!
